# 🐖 06 | Entrenamiento de Mask R-CNN para comparar con YOLO

**RECURSO: GPU T4 o superior**. Entrena solamente con train; selecciona pesos por rendimiento en val. No consulta el conjunto test. Guarda pesos, progreso y métricas.

**Proyecto:** Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖

Los notebooks son independientes: cada uno utiliza los productos guardados por la etapa anterior, sin repetir descarga o entrenamiento.


## Antes de ejecutar

**Ya existe el modelo YOLO entrenado: no volver a ejecutarlo.** Este notebook entrena una segunda arquitectura independiente: Mask R-CNN con ResNet50-FPN, inicializada con pesos preentrenados COCO. Sustituye los cabezales para dos clases: fondo y cerdo.

La comparación posterior será **exploratoria**: esta arquitectura, el presupuesto de cómputo y el preprocesamiento no son idénticos a YOLO. Para publicación, deberemos armonizar el evaluador y documentar el presupuesto.

Requisitos: datos `train/val/test` ya preparados por el notebook 02 y GPU T4 disponible. Las imágenes de test **no se cargan ni se evalúan** en este notebook.

**Duración:** este modelo puede tardar bastante más que YOLO11n. Por defecto ejecuta hasta 12 épocas, con evaluaciones sobre validación cada 2 épocas y guardado para continuar si Colab se interrumpe. No hay promesa de duración exacta.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Este notebook necesita GPU. Colab → Entorno de ejecución → Cambiar tipo → GPU T4.')
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl pillow opencv-python-headless tqdm


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, os

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError('No encuentro la carpeta del proyecto. Comprueba que Colab monta la cuenta correcta y que el nombre no cambió: '+str(ROOT))
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
EXTRACT=DATA/'extraido'/'FaroPigSeg'
SPLIT=DATA/'split_fecha_v1'
RES=ROOT/'03_Resultados'
EXP=ROOT/'05_Exportaciones_ZIP'
for s in ['01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento']:
    (RES/s).mkdir(parents=True,exist_ok=True)
EXP.mkdir(exist_ok=True,parents=True)
MANIFEST=SPLIT/'particion_fecha_manifest.json'
YAML=SPLIT/'data_fecha.yaml'
if not (MANIFEST.is_file() and YAML.is_file()):
    raise FileNotFoundError('Faltan los archivos de partición congelada; ejecutar notebook 02 en CPU antes de continuar.')
part=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert len(set(part['asignaciones_fecha'].values())-{'train','val','test'})==0
print('Carpeta del proyecto:',ROOT)
print('Fechas por grupo:',{k:sum(v==k for v in part['asignaciones_fecha'].values()) for k in ('train','val','test')})


In [ ]:
import numpy as np, pandas as pd,torch,cv2,yaml,random,math
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms.functional import to_tensor

class PigMasks(Dataset):
    def __init__(self,imagelist):
        self.paths=[Path(s.strip()) for s in Path(imagelist).read_text(encoding='utf8').splitlines() if s.strip()]
        if not self.paths:raise RuntimeError('La partición está vacía: '+str(imagelist))
        missing=[str(p) for p in self.paths if not p.is_file()]
        if missing: raise FileNotFoundError('Hay imágenes ausentes: '+str(missing[:3]))
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        rgb=np.asarray(Image.open(p).convert('RGB'))
        h,w=rgb.shape[:2]
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError('Etiqueta no localizada: '+str(lab))
        masks=[];boxes=[]
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip(): continue
            vals=row.split()
            if int(vals[0])!=0:raise ValueError('Clase inesperada en '+str(lab))
            points=np.asarray([float(v) for v in vals[1:]],dtype=np.float32)
            if len(points)<6 or len(points)%2:raise ValueError('Polígono inválido en '+str(lab))
            if not np.isfinite(points).all() or np.any((points<0)|(points>1)):
                raise ValueError('Coordenadas no normalizadas en '+str(lab))
            coords=points.reshape(-1,2)*np.asarray([w,h],dtype=np.float32)
            xy=np.rint(coords).astype(np.int32)
            xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[xy],1)
            yy,xx=np.where(mask)
            if len(xx)<3: continue
            x0,x1=int(xx.min()),int(xx.max()+1);y0,y1=int(yy.min()),int(yy.max()+1)
            if x1<=x0 or y1<=y0:continue
            masks.append(mask);boxes.append([x0,y0,x1,y1])
        if not masks:raise RuntimeError('Imagen sin máscaras válidas: '+p.name)
        tar={'boxes':torch.as_tensor(boxes,dtype=torch.float32),
             'labels':torch.ones((len(masks),),dtype=torch.int64),
             'masks':torch.as_tensor(np.stack(masks),dtype=torch.uint8),
             'image_id':torch.tensor([index],dtype=torch.int64),
             'area':torch.as_tensor([int(m.sum()) for m in masks],dtype=torch.float32),
             'iscrowd':torch.zeros((len(masks),),dtype=torch.int64)}
        return to_tensor(rgb.copy()),tar

def collate_fn(batch):return tuple(zip(*batch))

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_maskrcnn(pretrained=True):
    model=maskrcnn_resnet50_fpn(weights='DEFAULT' if pretrained else None,
                               weights_backbone=None,min_size=512,max_size=768,
                               trainable_backbone_layers=3)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,2)
    in_channels=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(in_channels,256,2)
    # Congelar los mismos niveles, tanto en arranque con pesos COCO como en reanudación sin descarga.
    for name,parameter in model.backbone.body.named_parameters():
        if name.startswith(('conv1.', 'bn1.', 'layer1.')):
            parameter.requires_grad_(False)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def config_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils
from contextlib import redirect_stdout
import io

def encode_mask(arr):
    enc=maskutils.encode(np.asfortranarray(np.asarray(arr,dtype=np.uint8)))
    enc['counts']=enc['counts'].decode('ascii')
    return enc

def coco_metric_for_model(model,dataset,device,desc='Validación',include_rows=False):
    model.eval();gts=[];preds=[];image_records=[];annotations=[];aid=1
    for i in range(len(dataset)):
        image,target=dataset[i]
        height,width=image.shape[1:]
        gts.append({'id':i+1,'width':width,'height':height,'file_name':dataset.paths[i].name})
        gt_masks=target['masks'].numpy()
        for m in gt_masks:
            encoded=encode_mask(m)
            x,y,w,h=[float(v) for v in maskutils.toBbox({'size':encoded['size'],'counts':encoded['counts'].encode('ascii')})]
            annotations.append({'id':aid,'image_id':i+1,'category_id':1,'segmentation':encoded,
                                'bbox':[x,y,w,h],'area':int(m.sum()),'iscrowd':0});aid+=1
        with torch.no_grad():
            out=model([image.to(device)])[0]
        scores=out['scores'].detach().cpu().numpy()
        ms=(out['masks'].detach().cpu().numpy()[:,0]>.5) if len(scores) else np.zeros((0,height,width),dtype=bool)
        for j,score in enumerate(scores):
            if score<0.001: continue
            m=ms[j].astype(np.uint8)
            if not m.any():continue
            enc=encode_mask(m)
            preds.append({'image_id':i+1,'category_id':1,'segmentation':enc,'score':float(score)})
        if include_rows:
            filtered=np.where(scores>=.25)[0]
            predm=[ms[k].astype(bool) for k in filtered if k<len(ms)]
            p_scores=[float(scores[k]) for k in filtered if k<len(ms)]
            pairs=[]
            for gi,g in enumerate(gt_masks):
                for pj,pm in enumerate(predm):
                    inter=np.logical_and(g,pm).sum();union=np.logical_or(g,pm).sum()
                    iou=inter/max(1,union)
                    if iou>=.5:pairs.append((float(iou),gi,pj))
            used_gt=set();used_pred=set()
            for _,gi,pj in sorted(pairs,reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi);used_pred.add(pj)
            tp=len(used_gt)
            image_records.append({'imagen':dataset.paths[i].name,
                'fecha':dataset.paths[i].name[:10],
                'anotaciones_GT':len(gt_masks),'detecciones_conf_025':len(predm),
                'TP_IoU05':tp,'FP_IoU05':len(predm)-tp,'FN_IoU05':len(gt_masks)-tp,
                'confianza_media_pred':np.mean(p_scores) if p_scores else np.nan})
        if (i+1)%50==0:print(desc,i+1,'/',len(dataset))
    gt=COCO()
    gt.dataset={'images':gts,'annotations':annotations,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
        dt=gt.loadRes(preds) if preds else gt.loadRes([])
        ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    return {'seg_map50_95':float(ev.stats[0]),'seg_map50':float(ev.stats[1]),
            'seg_map75':float(ev.stats[2]),'gt':len(annotations),
            'num_images':len(dataset),'pred_coarse':len(preds)},pd.DataFrame(image_records)


In [ ]:
import platform,sys,torchvision
import random,time
SEED=2026
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
# Ajustar si GPU da memoria insuficiente; batch=1 normalmente funciona en T4.
EPOCHS=12;BATCH=1;VAL_EVERY=2;PATIENCE_EVALS=3
LR=0.0001;WEIGHT_DECAY=0.0001
RUN=RES/'06_Entrenamiento'/'MaskRCNN_fecha_v1'
RUN.mkdir(parents=True,exist_ok=True)
BEST=RUN/'maskrcnn_best_model.pt'
CHECKPOINT=RUN/'maskrcnn_last_checkpoint.pt'
DONE=RUN/'ENTRENAMIENTO_FINALIZADO.json'
HISTORY=RUN/'historial_maskrcnn.csv'
assert not DONE.exists() or BEST.is_file(), 'Existe marca final pero no hay pesos. Revisa Drive.'
config=yaml.safe_load(YAML.read_text(encoding='utf8'))
train_ds=PigMasks(config['train']);val_ds=PigMasks(config['val'])
print('TRAIN',len(train_ds),'VAL',len(val_ds),'— test no se utiliza')
print('Comprobando primera imagen y máscara...')
img,tar=train_ds[0]; print(tuple(img.shape),'cerdos:',len(tar['masks']))
assert set(part['asignaciones_fecha'].values())=={'train','val','test'}
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,collate_fn=collate_fn)
device=torch.device('cuda:0')
model=build_maskrcnn(pretrained=not CHECKPOINT.exists() and not DONE.exists()).to(device)
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=WEIGHT_DECAY)
scaler=torch.amp.GradScaler('cuda',enabled=True)
start_epoch=1;best_ap=-1.0;no_improve=0
if CHECKPOINT.is_file():
    obj=torch.load(CHECKPOINT,map_location='cpu',weights_only=False)
    if obj.get('yaml_sha256')!=config_hash(YAML):raise RuntimeError('Cambió la partición; no mezclar entrenamientos.')
    model.load_state_dict(obj['model']);optimizer.load_state_dict(obj['optimizer'])
    scaler.load_state_dict(obj['scaler']);start_epoch=obj['epoch']+1
    best_ap=obj['best_ap'];no_improve=obj['no_improve']
    print('♻️ Reanudación desde época',start_epoch)
rows=pd.read_csv(HISTORY).to_dict('records') if HISTORY.is_file() else []
if DONE.is_file():
    print('✅ Modelo Mask R-CNN ya finalizado. No se repetirá el entrenamiento.')
else:
    for epoch in range(start_epoch,EPOCHS+1):
        t0=time.time();model.train();losses=[]
        for images,targets in train_loader:
            images=[x.to(device) for x in images]
            targets=[{k:v.to(device) for k,v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=True):
                ld=model(images,targets)
                loss=sum(ld.values())
            if not torch.isfinite(loss):raise RuntimeError('Pérdida no finita, revisar datos o LR')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.)
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.detach().cpu()))
        entry={'epoch':epoch,'train_loss_mean':float(np.mean(losses)),'seg_map50_95_val':np.nan,
               'seg_map50_val':np.nan,'duration_sec':round(time.time()-t0,2)}
        if epoch%VAL_EVERY==0 or epoch==EPOCHS:
            val_metrics,_=coco_metric_for_model(model,val_ds,device,desc='Validación Mask R-CNN')
            entry['seg_map50_95_val']=val_metrics['seg_map50_95'];entry['seg_map50_val']=val_metrics['seg_map50']
            ap=val_metrics['seg_map50_95']
            if ap>best_ap+0.0001:
                best_ap=ap;no_improve=0
                torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            'epoch':epoch,'best_ap':best_ap,'yaml_sha256':config_hash(YAML)},BEST)
                print('✅ Mejor modelo guardado, AP val:',round(best_ap,4))
            else:no_improve+=1
        rows.append(entry)
        pd.DataFrame(rows).to_csv(HISTORY,index=False)
        torch.save({'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
            'epoch':epoch,'best_ap':best_ap,'no_improve':no_improve,
            'yaml_sha256':config_hash(YAML)},CHECKPOINT)
        print('Época',epoch,'loss=',round(entry['train_loss_mean'],4),'val_AP=',entry['seg_map50_95_val'], 'tiempo(s)=',entry['duration_sec'])
        if no_improve>=PATIENCE_EVALS:
            print('Parada temprana: sin mejora en',PATIENCE_EVALS,'validaciones.');break
    assert BEST.is_file(),'No existe mejor modelo; revisar entrenamiento.'
    DONE.write_text(json.dumps({'fecha_utc':datetime.now(timezone.utc).isoformat(),
           'best_epoch':int(torch.load(BEST,map_location='cpu',weights_only=False)['epoch']),
           'best_val_seg_map50_95':best_ap,'yaml_sha256':config_hash(YAML),
           'torch':torch.__version__,'torchvision':torchvision.__version__,
           'epochs_max':EPOCHS,'batch':BATCH,'lr':LR,'seed':SEED,
           'gpu':torch.cuda.get_device_name(0),'nota':'No incluye test; preentrenamiento COCO.'},indent=2,ensure_ascii=False),encoding='utf-8')


In [ ]:
if HISTORY.is_file():
    d=pd.read_csv(HISTORY)
    p=RES/'03_Metricas'/'CERDOS_06_Historial_MaskRCNN.xlsx'
    with pd.ExcelWriter(p,engine='openpyxl') as w:d.to_excel(w,sheet_name='Epocas',index=False)
    print('✅ Historial Excel:',p)
print('✅ Pesos del mejor Mask R-CNN:',BEST,'\n✅ Registro:',DONE)
print('SIGUIENTE: Notebook 07 con GPU T4. No hace falta volver a entrenar YOLO.')


**Limitación:** entrenamiento y selección de pesos basados en validación; **ningún resultado de test se calcula aquí**. La evaluación independiente ocurre en notebook 07.
